# 05 RF-DETR b-case fallback crop pose 테스트

custom RF-DETR detector와 RF-DETR Keypoint Preview를 같은 frame에 적용한 뒤, custom detector만 잡은 b 케이스를 crop batch로 다시 pose 추론합니다. crop들은 한 장으로 merge하지 않고 list batch로 `RFDETRKeypointPreview.predict([...])`에 전달합니다.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").is_file() and (path / "src").is_dir():
            return path
    raise RuntimeError("repo root를 찾지 못했습니다. notebook을 repo 안에서 실행하거나 REPO_ROOT를 직접 지정하세요.")


REPO_ROOT = find_repo_root(Path.cwd())
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("REPO_ROOT:", REPO_ROOT)

from labelstudio_bbox_tools.pose_inference.fallback_rfdetr import run_rfdetr_pose_fallback_crop_test

## 설정

처음에는 `RUN_PREVIEW=False`, `RUN_INFERENCE=False`, `DRY_RUN=True`라서 아무 것도 실행하지 않습니다. 실제 테스트 전 `INPUT_PATH`, `OUT_DIR`, `DETECTOR_WEIGHTS`, class 설정을 채우세요.

In [ ]:
# ===== 사용자가 주로 바꾸는 설정 =====

INPUT_PATH = Path("/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260409_falldown_추론샘플영상(720p_10fps)_모음")
OUT_DIR = Path("/share_ssd/ltb/Users/ltb/박스_추론용_샘플영상들/260409_falldown_추론샘플영상(720p_10fps)_모음/pose_fallback_outputs")

# custom RF-DETR detection checkpoint입니다. 예: checkpoint_best_total.pth
DETECTOR_WEIGHTS = Path("/share_ssd/ltb/Users/ltb/251021_가중치_및_프로젝트아웃풋_정리_박스/260622_운영서버_도메인맞춤_추가학습_4차_전체merge_28cls_rfdetr_m640/rfdetr_medium_640_ms010_20260624_094203/checkpoint_best_total.pth")
DETECTOR_MODEL_VARIANT = "medium"  # "auto", "nano", "small", "medium", "large"

# 실제 실행 전 custom detector class YAML을 넣는 것을 권장합니다.
DETECTOR_CLASS_YAML = Path("/share_ssd/ltb/Users/ltb/251021_가중치_및_프로젝트아웃풋_정리_박스/260622_운영서버_도메인맞춤_추가학습_4차_전체merge_28cls_rfdetr_m640/rfdetr_medium_640_ms010_20260624_094203/data.yaml")
DETECTOR_MANUAL_CLASSES = []
TARGET_DETECTION_CLASSES = ["small_worker", "worker", "crouched_worker"]  # 예: ["person", "worker", "fall"]. 비우면 detector 결과 전체를 pose 후보로 사용합니다.

# None이면 RFDETRKeypointPreview official default pretrained weight를 사용합니다.
POSE_WEIGHTS = None
POSE_CLASS_YAML = None
POSE_MANUAL_CLASSES = []

DEVICE = "cuda"
DETECTOR_CONF = 0.25
DETECTOR_IOU = 0.50
POSE_CONF = 0.25
POSE_IOU = 0.50
KEYPOINT_CONF = 0.01
POSE_SHAPE = None  # 예: (576, 576). None이면 모델 기본 resolution 사용.

MATCH_IOU = 0.30
CROP_PADDING_RATIO = 0.15
MIN_CROP_SIZE = 32
FALLBACK_BATCH_SIZE = 8
MAX_FALLBACK_CROPS_PER_FRAME = None
MAX_POSE_PER_CROP = 1
FINAL_NMS_IOU = 0.50

RECURSIVE = False
MAX_VIDEOS = None
FRAME_STRIDE = 1
START_SECONDS = None
END_SECONDS = None
MAX_FRAMES = None

FONT_PATH = None
FONT_SIZE = 20
LINE_WIDTH = 3
KEYPOINT_RADIUS = 2
RUN_NAME = None
OVERWRITE = False

DRAW_BBOX = True
DRAW_SKELETON = True
DRAW_KEYPOINTS = True

# ===== b-case 시각화 / 디버그 저장 옵션 =====
# COLOR_BY_SOURCE=True이면 결과 source별로 색을 다르게 표시합니다.
# - matched_full_frame_pose: full-frame pose와 custom detector가 둘 다 잡은 객체
# - pose_only: pose model만 잡은 객체
# - fallback_crop_pose: b-case crop 재추론에 성공한 객체
# - fallback_failed_detection: b-case crop 재추론도 실패해서 custom detector bbox만 남은 객체
COLOR_BY_SOURCE = True
SOURCE_COLOR_MAP = None  # 예: {"fallback_failed_detection": (255, 40, 40)}. None이면 기본 색상 사용.
SOURCE_LABEL_PREFIXES = None  # None이면 fallback_pose/custom_detect prefix를 자동 사용.
DRAW_DETECTION_ONLY_BBOX = True  # b-case crop pose 실패 시 custom detector bbox를 custom_detect label로 그립니다.
DRAW_FALLBACK_SUCCESS_DASHED = True  # b-case crop pose 성공 bbox를 점선으로 그려 full-frame pose와 구분합니다.

# 디버그 이미지는 영상과 별도로 frame/crop jpg를 저장합니다. 많이 저장될 수 있으니 필요할 때만 켜세요.
SAVE_DEBUG_FRAMES = False  # True이면 debug_frames/fallback_success, debug_frames/fallback_failed에 frame jpg 저장.
SAVE_DEBUG_CROPS = False  # True이면 debug_crops/fallback_success, debug_crops/fallback_failed에 crop jpg 저장.
MAX_DEBUG_ITEMS_PER_CASE_PER_VIDEO = 200  # 영상 1개, case 1종류당 저장할 최대 jpg 수. 제한이 싫으면 None.
DEBUG_JPEG_QUALITY = 95


# ===== 안전 플래그 =====
RUN_PREVIEW = True
RUN_INFERENCE = True
DRY_RUN = False

## 1. Preview

영상 목록과 metadata만 확인합니다. 모델을 load하지 않습니다.

In [ ]:
if RUN_PREVIEW:
    preview = run_rfdetr_pose_fallback_crop_test(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        detector_weights=DETECTOR_WEIGHTS,
        detector_class_yaml=DETECTOR_CLASS_YAML,
        detector_manual_classes=DETECTOR_MANUAL_CLASSES,
        detector_model_variant=DETECTOR_MODEL_VARIANT,
        pose_weights=POSE_WEIGHTS,
        pose_class_yaml=POSE_CLASS_YAML,
        pose_manual_classes=POSE_MANUAL_CLASSES,
        target_detection_classes=TARGET_DETECTION_CLASSES,
        device=DEVICE,
        detector_conf=DETECTOR_CONF,
        detector_iou=DETECTOR_IOU,
        pose_conf=POSE_CONF,
        pose_iou=POSE_IOU,
        keypoint_conf=KEYPOINT_CONF,
        pose_shape=POSE_SHAPE,
        match_iou=MATCH_IOU,
        crop_padding_ratio=CROP_PADDING_RATIO,
        min_crop_size=MIN_CROP_SIZE,
        fallback_batch_size=FALLBACK_BATCH_SIZE,
        max_fallback_crops_per_frame=MAX_FALLBACK_CROPS_PER_FRAME,
        max_pose_per_crop=MAX_POSE_PER_CROP,
        final_nms_iou=FINAL_NMS_IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        color_by_source=COLOR_BY_SOURCE,
        source_color_map=SOURCE_COLOR_MAP,
        source_label_prefixes=SOURCE_LABEL_PREFIXES,
        draw_detection_only_bbox=DRAW_DETECTION_ONLY_BBOX,
        draw_fallback_success_dashed=DRAW_FALLBACK_SUCCESS_DASHED,
        save_debug_frames=SAVE_DEBUG_FRAMES,
        save_debug_crops=SAVE_DEBUG_CROPS,
        max_debug_items_per_case_per_video=MAX_DEBUG_ITEMS_PER_CASE_PER_VIDEO,
        debug_jpeg_quality=DEBUG_JPEG_QUALITY,
        overwrite=OVERWRITE,
        dry_run=True,
    )
    preview.as_dict()
else:
    print("RUN_PREVIEW=False 이므로 preview를 건너뜁니다.")

## 2. Fallback Crop Inference

실제 실행 전 `RUN_INFERENCE=True`, `DRY_RUN=False`로 바꿉니다. 결과에는 `predictions.jsonl`과 `fallback_cases_summary.json`이 생성됩니다. `SAVE_DEBUG_FRAMES=True`로 켜면 b-case 성공/실패 frame을 `debug_frames/` 하위 폴더에 따로 저장합니다.

In [ ]:
if RUN_INFERENCE:
    result = run_rfdetr_pose_fallback_crop_test(
        input_path=INPUT_PATH,
        out_dir=OUT_DIR,
        detector_weights=DETECTOR_WEIGHTS,
        detector_class_yaml=DETECTOR_CLASS_YAML,
        detector_manual_classes=DETECTOR_MANUAL_CLASSES,
        detector_model_variant=DETECTOR_MODEL_VARIANT,
        pose_weights=POSE_WEIGHTS,
        pose_class_yaml=POSE_CLASS_YAML,
        pose_manual_classes=POSE_MANUAL_CLASSES,
        target_detection_classes=TARGET_DETECTION_CLASSES,
        device=DEVICE,
        detector_conf=DETECTOR_CONF,
        detector_iou=DETECTOR_IOU,
        pose_conf=POSE_CONF,
        pose_iou=POSE_IOU,
        keypoint_conf=KEYPOINT_CONF,
        pose_shape=POSE_SHAPE,
        match_iou=MATCH_IOU,
        crop_padding_ratio=CROP_PADDING_RATIO,
        min_crop_size=MIN_CROP_SIZE,
        fallback_batch_size=FALLBACK_BATCH_SIZE,
        max_fallback_crops_per_frame=MAX_FALLBACK_CROPS_PER_FRAME,
        max_pose_per_crop=MAX_POSE_PER_CROP,
        final_nms_iou=FINAL_NMS_IOU,
        recursive=RECURSIVE,
        max_videos=MAX_VIDEOS,
        frame_stride=FRAME_STRIDE,
        start_seconds=START_SECONDS,
        end_seconds=END_SECONDS,
        max_frames=MAX_FRAMES,
        run_name=RUN_NAME,
        font_path=FONT_PATH,
        font_size=FONT_SIZE,
        line_width=LINE_WIDTH,
        keypoint_radius=KEYPOINT_RADIUS,
        draw_bbox=DRAW_BBOX,
        draw_skeleton=DRAW_SKELETON,
        draw_keypoints=DRAW_KEYPOINTS,
        color_by_source=COLOR_BY_SOURCE,
        source_color_map=SOURCE_COLOR_MAP,
        source_label_prefixes=SOURCE_LABEL_PREFIXES,
        draw_detection_only_bbox=DRAW_DETECTION_ONLY_BBOX,
        draw_fallback_success_dashed=DRAW_FALLBACK_SUCCESS_DASHED,
        save_debug_frames=SAVE_DEBUG_FRAMES,
        save_debug_crops=SAVE_DEBUG_CROPS,
        max_debug_items_per_case_per_video=MAX_DEBUG_ITEMS_PER_CASE_PER_VIDEO,
        debug_jpeg_quality=DEBUG_JPEG_QUALITY,
        overwrite=OVERWRITE,
        dry_run=DRY_RUN,
    )
    result.as_dict()
else:
    print("RUN_INFERENCE=False 이므로 실제 inference를 건너뜁니다.")